# Component B, Question 1: Multi-Agent Simulation Setup

**Cooperative task:** four e-puck robots search a 2 m × 2 m arena for a hidden
**signal source**, standing in for a gas leak or a trapped person's
transmitter. Each robot carries a simulated concentration sensor that reads
the signal only at its own position. No single robot can locate the source
by itself, so they must share what they measure.

| Item | Choice |
|---|---|
| World file | `webots/worlds/component_b_swarm.wbt` |
| Agents | 4 × e-puck (`robot_0` … `robot_3`) with 360° LiDAR (120 rays, 1 m), GPS, inertial unit, radio emitter/receiver, and a coloured collar |
| Supervisor | `swarm_supervisor`: runs the experiments, acts as the coordinator in centralised mode, injects faults, and records ground truth and collisions |
| Controllers | `swarm_controller` (identical on every robot; role from its name) and `swarm_supervisor` |
| Phases | ① **consensus** on a rendezvous point → ② **formation** around it → ③ **PSO search** for the source |

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import swarm_data as sd

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110})

## Agent roles and communication topology

All four robots run **the same controller** and are equal peers: there is no
leader. A robot's identity (0–3) only decides its formation slot and its
neighbours.

- **Distributed mode:** each robot exchanges messages only with its two neighbours on a **ring** (0–1–2–3–0). The radio physically broadcasts, so the topology is enforced by each receiver ignoring non-neighbours. That models a short-range network where robots can only reach a few peers. The ring is the sparsest topology that stays connected, and it still stays connected (as a line) if one robot fails.
- **Centralised mode:** the supervisor acts as a **coordinator** in a **star** topology. Robots send their data to it, and it broadcasts the combined answer back.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, mode in zip(axes, ["distributed (ring)", "centralised (star)"]):
    sd.draw_arena(ax)
    starts = np.array(sd.STARTS)
    if mode.startswith("distributed"):
        for i in range(sd.N_ROBOTS):
            j = (i + 1) % sd.N_ROBOTS
            ax.plot(*zip(starts[i], starts[j]), "k--", lw=1, alpha=0.6)
    else:
        ax.plot(0, 0, "ks", ms=10, label="coordinator")
        for s in starts:
            ax.plot(*zip(s, (0, 0)), "k--", lw=1, alpha=0.6)
    for i, (x, y) in enumerate(starts):
        ax.plot(x, y, "o", ms=14, color=sd.ROBOT_COLOURS[i], mec="k")
        ax.annotate(f"robot_{i}", (x, y), xytext=(0, 12),
                    textcoords="offset points", ha="center", fontsize=8)
    ax.set_title(f"Start positions and {mode} topology")
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(OUT_DIR / "b1_topology.png", bbox_inches="tight")
plt.show()

## Cooperative task objective and phases

| Phase | Time | What the robots do | Communication |
|---|---|---|---|
| ① Consensus | 0–6.25 s (25 rounds) | Agree on a rendezvous point: the average of their start positions | Exchange current estimate with neighbours each round |
| ② Formation | 6.25–26.25 s | Drive to a slot on a 0.25 m circle around the agreed point | None |
| ③ PSO search | 26.25–106.25 s (20 iterations × 4 s) | Sense the signal, then move as PSO particles towards the best positions found | Share personal-best position and value each iteration |

**Objective:** minimise the distance between the swarm's best-known position
(*gbest*) and the true source, without collisions, and keep working when an
agent or the coordinator fails.

## Experiments

Each press of Play runs **four experiment types**, each repeated with **three
random seeds** (12 runs). The seed changes PSO's random coefficients, and
Webots adds fresh LiDAR noise to every run, so repeats differ. All later
comparisons report mean ± standard deviation over the three runs, not a
single, possibly lucky, run.

| Experiment | Coordination | Fault injected at PSO iteration 5 |
|---|---|---|
| `distributed` | ring | none |
| `centralised` | coordinator | none |
| `distributed_robot_failure` | ring | robot_2 stops moving and goes silent |
| `centralised_coordinator_failure` | coordinator | coordinator stops responding |

## Example run: trajectories by phase

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
sd.draw_arena(ax)
styles = {"consensus": ":", "formation": "--", "pso": "-"}
for i in range(sd.N_ROBOTS):
    d = sd.robot_log("distributed_s1", i)
    for phase, style in styles.items():
        p = d[d.phase == phase]
        ax.plot(p.x, p.y, style, color=sd.ROBOT_COLOURS[i], lw=1.4,
                label=f"robot_{i}" if phase == "pso" else None)
    ax.plot(d.x.iloc[0], d.y.iloc[0], "o", color=sd.ROBOT_COLOURS[i], mec="k")
ax.set_title("Distributed run, seed 1: formation (dashed), then search (solid)")
ax.legend(fontsize=8, loc="lower right")
plt.savefig(OUT_DIR / "b1_trajectories.png", bbox_inches="tight")
plt.show()

## Communication load

In [ ]:
sd.kind_table(sd.KINDS, ["messages"])

Distributed coordination needs more messages: consensus needs 25 rounds of
neighbour exchanges, where the coordinator computes the average in one step.
Every message is local, though, and there is no single node that everything
depends on. Question 3 tests what that buys when something fails.

## Justification of the environment and controller design

- **Arena and obstacles.** Three box obstacles sit between the start positions and the source, so the robots must avoid obstacles *while* searching. The source sits in open space behind them, so finding it requires coordinated exploration rather than one robot driving straight to it.
- **Signal model.** The concentration is Gaussian, *exp(−d²/2σ²)* with σ = 0.5 m, plus sensor noise (σ = 0.01). That gives a smooth landscape that PSO can climb, and the noise makes single readings unreliable. Each robot's controller contains the source position only *inside the sensor model*; the search logic never uses it.
- **One controller for every robot.** This is homogeneous and scalable: adding a robot needs no new code, which is the defining property of a swarm.
- **Supervisor as a separate node.** Experiment management and ground-truth measurement are kept apart from the robots' decision-making, so the robots cannot "cheat" with global information. The supervisor also lets all experiments run from identical start states in one press of Play.
- **Collars at LiDAR height.** A first version produced repeated robot-to-robot collisions. The e-puck's LiDAR sits *above* the other robots' bodies, so they were invisible to each other. Each robot now wears a coloured collar at LiDAR height. A robot's own collar lies inside its LiDAR's near clipping plane (4.5 cm), so it is not seen. This is how real swarm robots are made mutually detectable.
- **Fixed random seeds.** PSO's random coefficients are seeded per robot and experiment, so runs are repeatable.